In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from sklearn.datasets import make_classification

X,y = make_classification(n_samples=10000, n_features=10, n_informative=3)
X

In [ ]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score
from sklearn.base import clone

def evaluateModelAccuracy(model):
    copyModel = clone(model)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Cross Val Score": (cross_val_score(copyModel, X,y,scoring='accuracy',cv=5)).mean()
    }

In [ ]:
from sklearn.tree import DecisionTreeClassifier
print("Decision Tree:", evaluateModelAccuracy( DecisionTreeClassifier() ))

# 1. Bagging (Row sampling With replacement )

#### a. Bagging using DT

In [ ]:
from sklearn.ensemble import BaggingClassifier
bag_dt = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=50,   # The number of base estimators in the ensemble.
    max_samples=0.5,   # row sampling
    bootstrap=True,    # with replacement
    random_state=42,
    verbose=1,
    n_jobs=1
)
print("DT with replacement:", evaluateModelAccuracy(bag_dt) )

#### b. Bagging using SVM

In [ ]:
from sklearn.svm import SVC
bag_svc = BaggingClassifier(
    estimator=SVC(),
    n_estimators=50,   # The number of base estimators in the ensemble.
    max_samples=0.25,   # row sampling
    bootstrap=True,    # with replacement
    random_state=42,
    verbose=1,
    n_jobs=1
)
print("SVC with replacement:", evaluateModelAccuracy(bag_svc) )

# 2. Pasting (Row sampling Without replacemnet)

#### a. Bagging using DT

In [ ]:
bag_dt = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=50, 
    max_samples=0.5,  
    bootstrap=False,  # without replacement
    random_state=42,
    verbose=1,
    n_jobs=1
)
print("DT without replacement:", evaluateModelAccuracy(bag_dt))

In [ ]:
#### b. Bagging using SVM
from sklearn.svm import SVC

bag_svc = BaggingClassifier(
    estimator=SVC(),
    n_estimators=50,  
    max_samples=0.25,  
    bootstrap=False,  # without replacement
    random_state=42,
    verbose=1,
    n_jobs=1
)
print("SVC without replacement:", evaluateModelAccuracy(bag_svc))

# 3. Random Subspace (col sampling + no row sampling)

#### a. Bagging using DT

In [ ]:
bag_dt = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=50,
    max_samples=1.0,  #(no row sampling)
    max_features=0.5, # col sampling
    random_state=42,
    verbose=1,
    n_jobs=1
)
print("DT RandomSubspace:", evaluateModelAccuracy(bag_dt))

#### b. Bagging using SVM

In [ ]:
bag_svc = BaggingClassifier(
    estimator=SVC(),
    n_estimators=50,
    max_samples=1.0,  #(no row sampling)
    max_features=0.25, # col sampling
    random_state=42,
    verbose=1,
    n_jobs=1
)
print("SVC RandomSubspace:", evaluateModelAccuracy(bag_svc))

# 4. Random Patches (col sampling + row sampling can (replacement / not))

#### a. Bagging using DT

In [ ]:
bag_dt = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=50,
    max_samples=0.25,  # row sampling
    bootstrap=True,  #(lets with replacement)
    max_features=0.5, # col sampling
    random_state=42,
    verbose=1,
    n_jobs=1
)
print("DT Random Patches:", evaluateModelAccuracy(bag_dt))

#### b. Bagging using SVM

In [ ]:
bag_svc = BaggingClassifier(
    estimator=SVC(),
    n_estimators=50,
    max_samples=0.25,  # row sampling
    bootstrap=False,  #(lets without replacement)
    max_features=0.25, # col sampling
    random_state=42,
    verbose=1,
    n_jobs=1
)
print("SVC Random Patches:", evaluateModelAccuracy(bag_svc))

# OOB Score

In [ ]:
bag_dt = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=50,
    max_samples=0.25, 
    bootstrap=True,  
    max_features=0.5, 
    oob_score=True,   # OOB Score
    random_state=42,
    verbose=1,
    n_jobs=1
)

bag_dt.fit(X_train, y_train)
print("OOB score:", bag_dt.oob_score_)

## Bagging Tips
- Bagging generally gives better results than Pasting
- Good results come around the 25% to 50% row sampling mark
- Random patches and subspaces should be used while dealing with high dimensional data
- To find the correct hyperparameter values we can do GridSearchCV/RandomSearchCV

# All possible case : Grid Search CV

In [ ]:
from sklearn.model_selection import GridSearchCV

params = [
    {
        'estimator': [DecisionTreeClassifier()],
        'n_estimators': [50,70],
        'max_samples': [0.7,1.0],
        'bootstrap': [True,False],
        'max_features': [0.4,0.7,1.0],
    },
    {
        'estimator': [SVC()],
        'n_estimators': [50,70],
        'max_samples': [0.7,1.0],
        'bootstrap': [True,False],
        'max_features': [0.4,0.7,1.0],
    }
]

In [ ]:
from sklearn.model_selection import ParameterGrid

grid = ParameterGrid(params)
print(len(grid))

In [ ]:
searchGrid = GridSearchCV(
    estimator=BaggingClassifier(n_jobs=-1),
    param_grid=params,
    cv=2,
    n_jobs=-1,
    verbose=1
)

In [ ]:
searchGrid.fit(X_train, y_train)
y_pred = searchGrid.predict(X_test)
accuracy_score(y_test, y_pred)

In [ ]:
len(searchGrid.cv_results_['params'])

In [ ]:
searchGrid.cv_results_['params']